# AWQ CLC Ablation on Colab

Fresh Colab bootstrap. It clones two GitHub branches into two folders, creates a `uv` venv in each folder, writes `.env`, installs requirements, and runs the 3-bit LLaMA-3.1-8B AWQ ablation.

- `no_k`: CLC without knee-point masking
- `no_f`: CLC without max-flip cap


## 1. Clone the two ablation branches

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hungtrab/smart-flip.git"
WORK_ROOT = Path("/content/clc_ablation")
BRANCHES = {
    "no_k": WORK_ROOT / "smart_flip_no_k",
    "no_f": WORK_ROOT / "smart_flip_no_f",
}

WORK_ROOT.mkdir(parents=True, exist_ok=True)

def get_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None

github_token = os.environ.get("GITHUB_TOKEN") or get_colab_secret("GITHUB_TOKEN")
clone_url = REPO_URL
if github_token:
    clone_url = REPO_URL.replace("https://", f"https://{github_token}@")

for branch, folder in BRANCHES.items():
    if folder.exists():
        shutil.rmtree(folder)
    print(f"Cloning branch {branch} into {folder}")
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, clone_url, str(folder)])


## 2. Create uv venvs and install requirements

In [ ]:
if shutil.which("uv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"])

for branch, folder in BRANCHES.items():
    print(f"Setting up venv for {branch}: {folder}")
    subprocess.check_call(["uv", "venv", "--system-site-packages", ".venv"], cwd=folder)
    python_bin = folder / ".venv" / "bin" / "python"
    subprocess.check_call([str(python_bin), "-m", "ensurepip", "--upgrade"], cwd=folder)
    subprocess.check_call([str(python_bin), "-m", "pip", "install", "-U", "pip"], cwd=folder)
    subprocess.check_call([str(python_bin), "-m", "pip", "install", "-r", "requirements.txt"], cwd=folder)


## 3. Create `.env` in each repo folder

In [ ]:
def collect_env_text():
    for path in [Path("/content/.env"), Path.cwd() / ".env"]:
        if path.exists():
            print(f"Using env file: {path}")
            return path.read_text()

    keys = [
        "HF_TOKEN",
        "HUGGINGFACE_HUB_TOKEN",
        "HF_HUB_TOKEN",
        "WANDB_API_KEY",
        "WANDB_ENTITY",
        "WANDB_PROJECT",
    ]
    lines = ["HF_HUB_DISABLE_XET=1"]
    for key in keys:
        value = os.environ.get(key) or get_colab_secret(key)
        if value:
            lines.append(f"{key}={value}")
            os.environ[key] = value
    if not any(line.startswith(("HF_TOKEN=", "HUGGINGFACE_HUB_TOKEN=", "HF_HUB_TOKEN=")) for line in lines):
        raise RuntimeError("Missing HF token. Put it in /content/.env, Colab Secrets, or os.environ.")
    if not any(line.startswith("WANDB_API_KEY=") for line in lines):
        raise RuntimeError("Missing WANDB_API_KEY. Put it in /content/.env, Colab Secrets, or os.environ.")
    return "\n".join(lines) + "\n"

env_text = collect_env_text()
for branch, folder in BRANCHES.items():
    env_path = folder / ".env"
    env_path.write_text(env_text)
    print(f"Wrote {env_path}")


## 4. Run the 3-bit ablation scripts

In [ ]:
for branch, folder in BRANCHES.items():
    python_bin = folder / ".venv" / "bin" / "python"
    env = os.environ.copy()
    env["PYTHON_BIN"] = str(python_bin)
    env.setdefault("MODEL_PATH", "meta-llama/Meta-Llama-3.1-8B")
    env.setdefault("WANDB_PROJECT", "smartflip_ablation")
    env.setdefault("USE_WANDB", "1")
    print(f"Running 3-bit ablation in {branch}: {folder}")
    subprocess.check_call(["bash", "scripts/bash/ablation/run_llama31_awq_ablation.sh", "3"], cwd=folder, env=env)


## 5. Terminal commands for 4-bit

In [ ]:
for branch, folder in BRANCHES.items():
    print(f"# {branch}")
    print(f"cd {folder} && PYTHON_BIN={folder}/.venv/bin/python bash scripts/bash/ablation/run_llama31_awq_ablation.sh 4")
